# 01 — Khám phá dữ liệu MovieLens-100K và MovieLens-1M theo thời gian

Việc 14 (plan §12). Notebook này mô tả hai bộ dữ liệu trước khi huấn luyện mô hình, tập trung vào **thời gian**:
rating được nhập như thế nào, chia theo mốc t1/t2 làm mất gì, và điều đó có ý nghĩa gì cho attention theo Δt của T-KGAT.

**Nguồn:** chỉ đọc `data/raw/` (dữ liệu gốc), `data/processed/` (split từ script 02) và `results/registry.csv`
qua `tkgat.registry.load()`. Mọi con số trong phần nhận xét được in ra từ code, không gõ tay.
Các hàm tính toán nằm trong `src/tkgat/analysis/eda.py` (có test trong `tests/test_eda.py`).

**Quy ước:**
- "Ngày đầu" của một user = **24 giờ kể từ rating đầu tiên** của user đó (in kèm số theo ngày lịch UTC để đối chiếu).
- Một **phiên** nhập rating kết thúc khi user nghỉ **quá 60 phút** (in kèm độ nhạy với ngưỡng 30 phút).
- Mục 1–5 dùng toàn bộ rating gốc; mục 3 (phần Δt), 6, 7 dùng split đã chia theo mốc.
- Độ lệch chuẩn qua seed là độ lệch chuẩn mẫu (`ddof=1`).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from tkgat import registry
from tkgat.analysis import eda
from tkgat.analysis.eda import DAY, HOUR
from tkgat.data.loaders import load

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# chart tokens (validated: scripts/validate_palette.js of the dataviz guide, light surface)
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
COLOR = {"ml-100k": "#2a78d6", "ml-1m": "#eb6834"}
LABEL = {"ml-100k": "ML-100K", "ml-1m": "ML-1M"}
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": AXIS,
        "axes.labelcolor": INK2,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.6,
        "axes.axisbelow": True,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "text.color": INK,
        "axes.titlecolor": INK,
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "lines.linewidth": 2,
        "legend.frameon": False,
        "figure.dpi": 110,
    }
)


def say(text):
    display(Markdown(text))


def pct(x, d=1):
    return f"{100 * x:.{d}f}%".replace(".", ",")


def num(x):
    return f"{x:,}".replace(",", ".")


def dec(x, d=2):
    return f"{x:.{d}f}".replace(".", ",")


def day(ts):
    return pd.to_datetime(ts, unit="s").strftime("%d/%m/%Y")


DATA = {}
for name, raw, proc in [
    ("ml-100k", "data/raw/ml-100k", "data/processed/ml-100k"),
    ("ml-1m", "data/raw/ml-1m", "data/processed/ml-1m"),
]:
    inter, items = load(name, ROOT / raw)
    meta = json.loads((ROOT / proc / "meta.json").read_text(encoding="utf-8"))
    train = pd.read_csv(ROOT / proc / "train.csv")
    DATA[name] = {"inter": inter, "items": items, "meta": meta, "train": train}
for name, d in DATA.items():
    print(f"{LABEL[name]}: {len(d['inter']):,} ratings gốc, {len(d['train']):,} positive train")

## 1. Tổng quan

In [ ]:
ov = pd.DataFrame({LABEL[n]: eda.overview(d["inter"]) for n, d in DATA.items()})
display(ov)
years = {n: eda.year_shares(d["inter"]) for n, d in DATA.items()}
display(pd.DataFrame({LABEL[n]: y for n, y in years.items()}).fillna(0).map(pct).rename_axis("năm"))

clean = DATA["ml-100k"]["meta"]["clean"]
o1, o100 = ov[LABEL["ml-1m"]], ov[LABEL["ml-100k"]]
y1 = years["ml-1m"]
peak_year = int(y1.idxmax())
m1 = DATA["ml-1m"]["meta"]
last1 = DATA["ml-1m"]["inter"]["ts"].max()
later = ", ".join(str(y) for y in y1.index if y > peak_year)
say(f"""
- ML-1M lớn hơn ML-100K khoảng {o1["ratings"] / o100["ratings"]:.0f} lần về số rating, có {num(o1["users"])}
  user so với
  {num(o100["users"])}, nhưng thưa hơn (mật độ {pct(o1["density"])} so với {pct(o100["density"])}).
- ML-100K chỉ kéo dài {o100["span_days"]} ngày ({o100["first"]} → {o100["last"]}). ML-1M kéo dài tới
  {o1["last"]:%m/%Y} ({num(o1["span_days"])} ngày), **nhưng {pct(y1.max())} rating nằm trong năm {peak_year}**,
  phần còn lại
  rải mỏng trong các năm {later}. Vì mốc chia là phân vị thời gian, t1 = {day(m1["t1"])} và t2 =
  {day(m1["t2"])} của ML-1M
  đều rơi vào cuối năm {peak_year}: cửa sổ val chỉ dài {(m1["t2"] - m1["t1"]) / DAY:.0f} ngày còn test kéo dài
  {dec((last1 - m1["t2"]) / DAY / 365.25, 1)} năm (mục 2 và 6).
- ML-100K được làm sạch trước khi chia: bỏ item {", ".join(map(str, clean["drop_untitled"]["untitled_items"]))}
  ({clean["drop_untitled"]["interactions_dropped"]} rating) và gộp
  {clean["merge_duplicate_titles"]["duplicate_title_items"]}
  item trùng tiêu đề ({clean["merge_duplicate_titles"]["interactions_remapped"]} rating được gán lại). Mục 1–5
  dùng
  dữ liệu gốc nên các số này chưa trừ phần làm sạch.
""")

## 2. Số rating theo tuần và hai mốc t1, t2

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
weekly = {}
for ax, (name, d) in zip(axes, DATA.items(), strict=True):
    w = weekly[name] = eda.weekly_counts(d["inter"])
    t1, t2 = (pd.to_datetime(d["meta"][k], unit="s") for k in ("t1", "t2"))
    end = w.index.max() + pd.Timedelta(days=7)
    ax.axvspan(t1, t2, color=GRID, alpha=0.8, lw=0)
    ax.axvspan(t2, end, color=GRID, alpha=0.4, lw=0)
    ax.plot(w.index, w.values, color=COLOR[name])
    for t, lab, ha in ((t1, "t1 ", "right"), (t2, " t2", "left")):
        ax.axvline(t, color=INK2, lw=1, ls="--")
        ax.text(t, ax.get_ylim()[1] * 0.97, lab, color=INK2, va="top", ha=ha, fontsize=9)
    ax.set_title(f"{LABEL[name]}: số rating mỗi tuần")
    ax.set_ylabel("rating / tuần")
    ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()


def window_days(meta):
    return (meta["t2"] - meta["t1"]) / DAY


def peak(name):
    w = weekly[name]
    return w.idxmax(), int(w.max()), w.max() / w.median()


m100 = DATA["ml-100k"]["meta"]
cfg = m1["config"]
p100, p1 = peak("ml-100k"), peak("ml-1m")
say(f"""
Vùng xám đậm là cửa sổ val [t1, t2), vùng xám nhạt là test (≥ t2). Mỗi cửa sổ chứa
{pct(cfg["q_test"] - cfg["q_val"], 0)} số rating (phân vị {dec(cfg["q_val"], 1)} và {dec(cfg["q_test"], 1)}).
- Số rating mỗi tuần **dao động mạnh**, theo từng đợt user mới đăng ký: tuần đông nhất của ML-100K
  ({p100[0]:%d/%m/%Y}) có {num(p100[1])} rating, gấp {dec(p100[2], 1)} lần tuần trung vị; của ML-1M
  ({p1[0]:%d/%m/%Y})
  có {num(p1[1])}, gấp {dec(p1[2], 0)} lần.
- ML-100K: val dài {window_days(m100):.0f} ngày, test
  {(DATA["ml-100k"]["inter"]["ts"].max() - m100["t2"]) / DAY:.0f} ngày,
  hai cửa sổ ngắn tương đương nhau.
- ML-1M: val chỉ dài {window_days(m1):.0f} ngày ({day(m1["t1"])} → {day(m1["t2"])}), còn test kéo dài
  {num(int((last1 - m1["t2"]) / DAY))} ngày vì sau đó rating rất thưa: hai cửa sổ có cùng số rating nhưng độ
  dài thời gian
  khác hẳn nhau.
""")

## 3. Rating được nhập theo phiên

In [ ]:
rows = {}
for name, d in DATA.items():
    x = d["inter"]
    s60, s30 = eda.sessions(x, gap_s=HOUR), eda.sessions(x, gap_s=30 * 60)
    days = eda.active_days(x)
    rows[LABEL[name]] = {
        "rating trong 24 giờ đầu": eda.first_window_share(x),
        "rating cùng ngày lịch UTC với rating đầu": eda.first_calendar_day_share(x),
        "user chỉ hoạt động 1 ngày": float((days == 1).mean()),
        "số phiên (nghỉ > 60 phút)": len(s60),
        "trung vị số phiên / user": float(s60.groupby("user").size().median()),
        "trung vị thời lượng phiên ≥ 2 rating (phút)": float(s60.loc[s60.n_ratings > 1, "duration_s"].median() / 60),
        "số phiên (nghỉ > 30 phút)": len(s30),
        "_s60": s60,
    }
table = pd.DataFrame({k: {c: v for c, v in r.items() if not c.startswith("_")} for k, r in rows.items()})
display(table)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1, 1.6]})
ax = axes[0]
names = list(DATA)
vals = [rows[LABEL[n]]["rating trong 24 giờ đầu"] for n in names]
ax.barh([LABEL[n] for n in names], vals, color=[COLOR[n] for n in names], height=0.5, edgecolor=SURFACE, linewidth=2)
for i, v in enumerate(vals):
    ax.text(v + 0.01, i, pct(v), va="center", color=INK, fontsize=10)
ax.set_xlim(0, 1)
ax.set_title("Tỷ lệ rating trong 24 giờ đầu của user")
ax.grid(axis="y", visible=False)

ax = axes[1]
for name in names:
    s = rows[LABEL[name]]["_s60"]
    dur = np.sort(s.loc[s.n_ratings > 1, "duration_s"].to_numpy() / 60)
    ax.plot(dur, np.arange(1, len(dur) + 1) / len(dur), color=COLOR[name], label=LABEL[name])
ax.set_xscale("log")
ax.set_xlabel("thời lượng phiên (phút, thang log)")
ax.set_ylabel("tỷ lệ phiên ≤ x")
ax.set_title("Phân bố thời lượng phiên (phiên có ≥ 2 rating)")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

r100, r1 = rows[LABEL["ml-100k"]], rows[LABEL["ml-1m"]]
say(f"""
- **{pct(r100["rating trong 24 giờ đầu"])}** rating của ML-100K và **{pct(r1["rating trong 24 giờ đầu"])}** của
  ML-1M được nhập
  trong 24 giờ kể từ rating đầu tiên của user (nếu tính theo ngày lịch UTC:
  {pct(r100["rating cùng ngày lịch UTC với rating đầu"])}
  và {pct(r1["rating cùng ngày lịch UTC với rating đầu"])}). {pct(r100["user chỉ hoạt động 1 ngày"])} user của
  ML-100K và
  {pct(r1["user chỉ hoạt động 1 ngày"])} của ML-1M chỉ hoạt động đúng một ngày.
- Một phiên (có ít nhất 2 rating) có trung vị thời lượng
  {dec(r100["trung vị thời lượng phiên ≥ 2 rating (phút)"], 1)} phút
  (ML-100K) và {dec(r1["trung vị thời lượng phiên ≥ 2 rating (phút)"], 1)} phút (ML-1M), và trung vị mỗi user
  chỉ có
  {r100["trung vị số phiên / user"]:.0f} phiên. Đổi ngưỡng nghỉ từ 60 xuống 30 phút chỉ tăng số phiên từ
  {num(r100["số phiên (nghỉ > 60 phút)"])} lên {num(r100["số phiên (nghỉ > 30 phút)"])} (ML-100K) và từ
  {num(r1["số phiên (nghỉ > 60 phút)"])} lên {num(r1["số phiên (nghỉ > 30 phút)"])} (ML-1M), nên kết luận không
  phụ thuộc ngưỡng.
- Timestamp của MovieLens là **lúc nhập rating**, không phải lúc xem phim: người dùng đăng ký rồi chấm hàng
  loạt phim đã xem từ trước.
""")

### 3b. Δt giữa các cạnh của cùng một node (hệ quả cho T-KGAT)

T-KGAT trừ `γ·δ_r·Δt` vào điểm attention **trước softmax**, và softmax chỉ so sánh các cạnh **của cùng một node**.
Vì vậy điều quan trọng không phải Δt trung bình của cả bộ dữ liệu mà là các cạnh của một node có trải ra theo thời gian hay không.
Tính trên các positive của **train** (trước t1), cho cả phía user (các item user đã rate) và phía item (các user đã rate item đó).

In [ ]:
TH = {"< 1 giờ": HOUR, "< 1 ngày": DAY, "< 30 ngày": 30 * DAY}
spread_rows, spread = {}, {}
for name, d in DATA.items():
    tr, t1 = d["train"], d["meta"]["t1"]
    gaps = eda.consecutive_gaps(tr, by="user")
    for by in ("user", "item"):
        sp = eda.node_time_spread(tr, by=by, ref=t1)
        spread[(name, by)] = sp
        row = {f"độ trải {k}": v for k, v in eda.share_below(sp["spread_s"], TH).items()}
        row["trung vị std(t1 − ts) (ngày)"] = float(sp["std_days"].median())
        row["số node"] = len(sp)
        spread_rows[(LABEL[name], by)] = row
    spread_rows[(LABEL[name], "user")]["Δt liên tiếp < 1 phút"] = float((gaps < 60).mean())
    spread_rows[(LABEL[name], "user")]["Δt liên tiếp < 1 giờ"] = float((gaps < HOUR).mean())
sp_table = pd.DataFrame(spread_rows).T
sp_table.index.names = ["dataset", "node"]
display(sp_table)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
floor = 1 / 1440  # one minute, in days: spreads of 0 are drawn at this floor on the log axis
for ax, by, title in (
    (axes[0], "user", "Phía user: các cạnh train của một user"),
    (axes[1], "item", "Phía item: các cạnh train của một item"),
):
    for name in DATA:
        v = np.sort(np.maximum(spread[(name, by)]["spread_s"].to_numpy() / DAY, floor))
        ax.plot(v, np.arange(1, len(v) + 1) / len(v), color=COLOR[name], label=LABEL[name])
    for t, lab in ((1 / 24, "1 giờ"), (1, "1 ngày"), (30, "30 ngày")):
        ax.axvline(t, color=MUTED, lw=0.8, ls=":")
        ax.text(t, 0.02, f" {lab}", color=MUTED, fontsize=8)
    ax.set_xscale("log")
    ax.set_xlim(floor, 1500)
    ax.set_xlabel("độ trải max(ts) − min(ts) (ngày, thang log; 0 vẽ tại 1 phút)")
    ax.set_title(title)
axes[0].set_ylabel("tỷ lệ node có độ trải ≤ x")
axes[1].legend(loc="upper left")
fig.tight_layout()
plt.show()


def g(name, by, col):
    return sp_table.loc[(LABEL[name], by), col]


say(f"""
- **Phía user:** {pct(g("ml-100k", "user", "độ trải < 1 giờ"))} user của ML-100K và
  {pct(g("ml-1m", "user", "độ trải < 1 giờ"))}
  của ML-1M có toàn bộ cạnh train nằm trong **một giờ**; {pct(g("ml-100k", "user", "Δt liên tiếp < 1 phút"))} /
  {pct(g("ml-1m", "user", "Δt liên tiếp < 1 phút"))} khoảng cách giữa hai rating liên tiếp của một user ngắn
  hơn một phút.
  Trung vị độ lệch chuẩn của (t1 − ts) trong một user chỉ khoảng
  {g("ml-100k", "user", "trung vị std(t1 − ts) (ngày)") * 1440:.0f} /
  {g("ml-1m", "user", "trung vị std(t1 − ts) (ngày)") * 1440:.0f} phút.
- **Phía item:** gần như không item nào có các cạnh gom trong 30 ngày
  ({pct(g("ml-100k", "item", "độ trải < 30 ngày"))} /
  {pct(g("ml-1m", "item", "độ trải < 30 ngày"))}); trung vị độ lệch chuẩn là
  {g("ml-100k", "item", "trung vị std(t1 − ts) (ngày)"):.0f} /
  {g("ml-1m", "item", "trung vị std(t1 − ts) (ngày)"):.0f} ngày, vì các user đến hệ thống vào những thời điểm
  khác nhau.
- **Hệ quả:** trong softmax của một user, các Δt gần như bằng nhau nên phép trừ `γ·δ_r·Δt` gần như không đổi
  thứ tự
  attention: suy giảm theo thời gian **hầu như không có tác dụng ở phía user**. Ở phía item, Δt trải rộng nên
  suy giảm
  **có thể có tác dụng**. Đây là giả thuyết cho RQ3, chưa phải kết luận; sẽ kiểm bằng thí nghiệm hoán vị
  timestamp (bắt buộc
  cho mọi biến thể có thời gian).
""")

## 4. Phân bố giá trị rating

In [ ]:
dist = pd.DataFrame({LABEL[n]: eda.rating_distribution(d["inter"]) for n, d in DATA.items()})
fig, ax = plt.subplots(figsize=(6.5, 3.2))
width = 0.38
for k, name in enumerate(DATA):
    xs = dist.index + (k - 0.5) * width
    ax.bar(xs, dist[LABEL[name]], width=width, color=COLOR[name], label=LABEL[name], edgecolor=SURFACE, linewidth=2)
ax.set_xticks(dist.index)
ax.set_xlabel("rating")
ax.set_ylabel("tỷ lệ rating")
ax.set_title("Phân bố giá trị rating")
ax.legend()
ax.grid(axis="x", visible=False)
fig.tight_layout()
plt.show()
display(dist.map(pct))
pos = {n: float((d["inter"]["rating"] >= 4).mean()) for n, d in DATA.items()}
mode = {n: int(dist[LABEL[n]].idxmax()) for n in DATA}
say(f"""
- Rating lệch về phía cao: mức phổ biến nhất là {mode["ml-100k"]} ở ML-100K và {mode["ml-1m"]} ở ML-1M
  ({pct(dist[LABEL["ml-100k"]].max())} và {pct(dist[LABEL["ml-1m"]].max())} số rating). Tỷ lệ rating ≥ 4 (được
  coi là positive) là
  **{pct(pos["ml-100k"])}** ở ML-100K và **{pct(pos["ml-1m"])}** ở ML-1M.
- Hai bộ có phân bố gần giống nhau, nên ngưỡng "rating ≥ 4" cho tỷ lệ positive tương đương và kết quả hai bộ so
  sánh được.
""")

## 5. Năm phát hành phim so với thời điểm rating

In [ ]:
ages = {n: eda.film_age_at_rating(d["inter"], d["items"]) for n, d in DATA.items()}
fig, ax = plt.subplots(figsize=(8, 3.2))
bins = np.arange(-1, 62) - 0.5
for name, a in ages.items():
    h, edges = np.histogram(a.clip(upper=60), bins=bins)
    ax.step(edges[:-1] + 0.5, h / len(a), where="mid", color=COLOR[name], label=LABEL[name])
ax.set_xlabel("tuổi phim lúc được rate (năm rating − năm phát hành; ≥ 60 gộp tại 60)")
ax.set_ylabel("tỷ lệ rating")
ax.set_title("Phim được rate khi bao nhiêu tuổi")
ax.legend()
fig.tight_layout()
plt.show()
missing_year = int(DATA["ml-100k"]["items"]["year"].isna().sum())
say(f"""
- Trung vị tuổi phim lúc được rate là **{ages["ml-100k"].median():.0f} năm** ở ML-100K và
  **{ages["ml-1m"].median():.0f} năm** ở ML-1M;
  phim mới (≤ 1 năm) chiếm {pct(float((ages["ml-100k"] <= 1).mean()))} và
  {pct(float((ages["ml-1m"] <= 1).mean()))} số rating.
- Một nửa số rating dành cho phim đã ra mắt từ {ages["ml-100k"].median():.0f} năm (ML-100K) và
  {ages["ml-1m"].median():.0f} năm
  (ML-1M) trở lên: người dùng chấm cả phim đã xem từ lâu khi đăng ký, khớp với hành vi nhập theo phiên ở mục 3.
  Thời điểm rating vì vậy không cho biết lúc người dùng xem phim.
- Năm phát hành lấy từ tiêu đề (ML-1M không có cột ngày phát hành); {missing_year} item của ML-100K không có
  năm và bị bỏ qua.
""")

## 6. Hệ quả của chia theo mốc: tương tác val/test bị loại vì user hoặc item mới

In [ ]:
losses = pd.concat({LABEL[n]: eda.split_losses(d["meta"]) for n, d in DATA.items()}, names=["dataset", "part"])
display(losses)

PARTS = {
    "kept": ("còn lại", "#1baf7a"),
    "dropped_cold_user": ("loại: user mới", "#eda100"),
    "dropped_cold_item": ("loại: item mới", "#e87ba4"),
}
fig, ax = plt.subplots(figsize=(8.5, 3))
labels = [f"{ds} · {p}" for ds, p in losses.index]
left = np.zeros(len(losses))
for col, (lab, color) in PARTS.items():
    share = (losses[col] / losses["positives_before_filter"]).to_numpy()
    ax.barh(labels, share, left=left, color=color, label=lab, height=0.55, edgecolor=SURFACE, linewidth=2)
    left += share
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel("tỷ lệ trong số positive của cửa sổ (trước khi lọc)")
ax.set_title("Positive val/test còn lại và bị loại")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.55), ncol=3)
ax.grid(axis="y", visible=False)
fig.tight_layout()
plt.show()

L = losses


def v(ds, p, c):
    return L.loc[(LABEL[ds], p), c]


say(f"""
- **ML-1M:** val chỉ còn **{num(int(v("ml-1m", "val", "kept")))}** positive
  ({pct(v("ml-1m", "val", "kept_share"))}) vì
  {num(int(v("ml-1m", "val", "dropped_cold_user")))} positive thuộc user chưa có trong train; test còn
  **{num(int(v("ml-1m", "test", "kept")))}** ({pct(v("ml-1m", "test", "kept_share"))}), chỉ mất
  {num(int(v("ml-1m", "test", "dropped_cold_user")))} vì user mới. Lý do: cửa sổ val ngắn
  ({window_days(m1):.0f} ngày,
  {day(m1["t1"])} → {day(m1["t2"])}) chứa chủ yếu user mới đăng ký, còn cửa sổ test dài
  {dec((last1 - m1["t2"]) / DAY / 365.25, 1)} năm chứa nhiều user cũ quay lại.
- **ML-100K:** cả hai cửa sổ đều mất phần lớn positive vì user mới (val còn
  {pct(v("ml-100k", "val", "kept_share"))},
  test còn {pct(v("ml-100k", "test", "kept_share"))}), chỉ còn {int(v("ml-100k", "val", "users_kept"))} user
  val và
  {int(v("ml-100k", "test", "users_kept"))} user test. Đây là cái giá của chia theo mốc chung khi rating được
  nhập theo phiên;
  vì vậy kết luận nên dựa chủ yếu vào ML-1M ({num(int(v("ml-1m", "test", "users_kept")))} user test).
- Item mới làm mất rất ít ({int(v("ml-1m", "test", "dropped_cold_item"))} positive test ở ML-1M).
""")

## 7. Độ phổ biến: đuôi dài và MostPop gần đây

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.6))
for name, d in DATA.items():
    c = eda.popularity_curve(d["train"])
    ax.plot(c["item_share"], c["cum_share"], color=COLOR[name], label=LABEL[name])
ax.plot([0, 1], [0, 1], color=MUTED, lw=1, ls="--")
ax.text(0.5, 0.56, "phân bố đều", color=MUTED, fontsize=8, rotation=33, ha="left", va="top")
ax.set_xlabel("tỷ lệ item (xếp theo số positive train, giảm dần)")
ax.set_ylabel("tỷ lệ positive train cộng dồn")
ax.set_title("Đuôi dài của độ phổ biến (train)")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()
top = {n: (eda.top_share(d["train"], 0.1), eda.top_share(d["train"], 0.2)) for n, d in DATA.items()}

reg = registry.load(ROOT / "results" / "registry.csv")
base = eda.baseline_table(reg)
base = base[base["model"].isin(["mostpop", "mostpop_recent", "bprmf"])]
display(base)


def recall(ds, model):
    r = base[(base["dataset"] == ds) & (base["model"] == model)]
    return None if r.empty else float(r["mean"].iloc[0])


lines = []
for ds in DATA:
    mp, rec = recall(ds, "mostpop"), recall(ds, "mostpop_recent")
    if mp is None or rec is None:
        lines.append(f"- {LABEL[ds]}: **chưa có** baseline chính thức trong registry (việc 17).")
    else:
        win = json.loads(base[(base["dataset"] == ds) & (base["model"] == "mostpop_recent")]["config"].iloc[0])
        lines.append(
            f"- {LABEL[ds]}: MostPop {win['window_days']} ngày gần nhất đạt test Recall@20 = {dec(rec, 4)}, so với "
            f"{dec(mp, 4)} của MostPop trên toàn bộ train ({'+' if rec >= mp else '−'}{pct(abs(rec / mp - 1), 0)})."
        )
say(
    f"""
- 10% item phổ biến nhất nhận {pct(top["ml-100k"][0])} positive train ở ML-100K và {pct(top["ml-1m"][0])} ở
  ML-1M
  (20% item: {pct(top["ml-100k"][1])} và {pct(top["ml-1m"][1])}): độ phổ biến tập trung mạnh,
  {"ML-1M" if top["ml-1m"][0] > top["ml-100k"][0] else "ML-100K"} tập trung hơn.
"""
    + "\n".join(lines)
    + """

Độ phổ biến **gần đây** là tín hiệu thời gian đơn giản nhất: nếu nó đã thắng độ phổ biến toàn bộ
thì mô hình có thời gian phải vượt được MostPop gần đây, không chỉ MostPop.
(Số đọc từ `registry.load()`, độ lệch chuẩn là sample std qua seed.)
"""
)

## 8. Tóm tắt

In [ ]:
say(f"""
1. Timestamp của MovieLens là lúc **nhập** rating: {pct(r100["rating trong 24 giờ đầu"])} (ML-100K) và
   {pct(r1["rating trong 24 giờ đầu"])} (ML-1M) rating nằm trong 24 giờ đầu của user.
2. Vì vậy Δt giữa các cạnh của **một user** gần như bằng 0 ({pct(g("ml-1m", "user", "độ trải < 1 giờ"))} user
   ML-1M trải dưới
   một giờ), còn các cạnh của **một item** trải hàng tháng (trung vị std
   {g("ml-1m", "item", "trung vị std(t1 − ts) (ngày)"):.0f} ngày ở ML-1M).
   Giả thuyết cho RQ3: suy giảm theo Δt chỉ có thể có tác dụng ở phía item; kiểm bằng thí nghiệm hoán vị
   timestamp.
3. Chia theo mốc chung làm mất phần lớn positive val/test của ML-100K (còn
   {int(v("ml-100k", "test", "users_kept"))} user test);
   ML-1M giữ {num(int(v("ml-1m", "test", "users_kept")))} user test nên là bộ chính để kết luận.
4. ML-1M kéo dài tới {o1["last"]:%m/%Y} nhưng {pct(y1.max())} rating nằm trong năm {peak_year}: cửa sổ val
   ngắn, test dài.
5. Rating ≥ 4 chiếm {pct(pos["ml-100k"])} / {pct(pos["ml-1m"])}; độ phổ biến có đuôi dài và MostPop gần đây là
   baseline thời gian cần vượt.
""")